In [ ]:
import torch
from PIL import Image, ImageFilter, ImageDraw
from diffusers import StableDiffusionInpaintPipeline, AutoencoderKL
from torch.utils.tensorboard import SummaryWriter
import datetime

state = {
    "shift": (0, 0), # memorizzo dx e dy per unrolling
    "has_replicated": False # sono al 60%? -> Latent replication
}

def callback(pipe, step_index, timestep, callback_kwargs):
    global state
    latents = callback_kwargs["latents"]
    mask = callback_kwargs["mask_ins"]
    masked_image_latents = callback_kwargs["masked_image_latents"]

    # --- EQUAZIONE 7: UNROLLING ---
    # Se avevamo uno shift attivo dallo step precedente, lo annulliamo.
    # Questo riporta l'immagine "al centro" per mantenere la coerenza globale.
    if state["current_shift"] != (0, 0):
        sx, sy = state["current_shift"]
        latents = torch.roll(latents, shifts=(-sy, -sx), dims=(-2, -1))
        mask = torch.roll(mask, shifts=(-sy, -sx), dims=(-2, -1))
        masked_image_latents = torch.roll(masked_image_latents, shifts=(-sy, -sx), dims=(-2, -1))

    # --- EQUAZIONE 6: ROLLING ---
    # Calcoliamo un nuovo spostamento casuale per lo step successivo.
    # Non lo facciamo all'ultimo step, così l'immagine finale è già "srotolata".
    if step_index < len(pipe.scheduler.timesteps) - 1:
        new_sx = torch.randint(0, latents.shape[-1], (1,)).item()
        new_sy = torch.randint(0, latents.shape[-2], (1,)).item()
        
        # Spostiamo tutto in sincronia: rumore + maschera + immagine guida
        latents = torch.roll(latents, shifts=(new_sy, new_sx), dims=(-2, -1))
        mask = torch.roll(mask, shifts=(new_sy, new_sx), dims=(-2, -1))
        masked_image_latents = torch.roll(masked_image_latents, shifts=(new_sy, new_sx), dims=(-2, -1))
        
        state["current_shift"] = (new_sx, new_sy)
    else:
        state["current_shift"] = (0, 0)

    # Aggiorniamo i tensori nella pipeline
    callback_kwargs["latents"] = latents
    callback_kwargs["mask_ins"] = mask
    callback_kwargs["masked_image_latents"] = masked_image_latents

    return callback_kwargs

In [ ]:
log_dir = "runs/esperimento_" + datetime.datetime.now().strftime("%Y%m%d-%H%M%S")
writer = SummaryWriter(log_dir)

device = "cuda"
model_id = "sd-legacy/stable-diffusion-inpainting"
adapter_id = "h94/IP-Adapter"

In [ ]:
vae = AutoencoderKL.from_pretrained(
    "stabilityai/sd-vae-ft-mse",
    torch_dtype=torch.float16
).to(device)

pipe = StableDiffusionInpaintPipeline.from_pretrained(
    model_id, 
    use_safetensors=False,
    torch_dtype=torch.float16,
    safety_checker=None,
    requires_safety_checker=False,
    vae = vae
).to(device)

#pipe.enable_model_cpu_offload()
#pipe.enable_attention_slicing()

#pipe.vae.enable_tiling()

In [ ]:
def patch_conv_circular(model):
    for layer in model.modules():
        if isinstance(layer, torch.nn.Conv2d):
            layer.padding_mode = 'circular'

patch_conv_circular(pipe.unet)

In [ ]:
from diffusers import DDIMScheduler
pipe.scheduler = DDIMScheduler.from_config(pipe.scheduler.config)

In [ ]:
pipe.load_ip_adapter(adapter_id, subfolder="models", weight_name="ip-adapter_sd15.bin")
pipe.set_ip_adapter_scale(0.8)

#print(torch.cuda.memory.memory_summary(device=device, abbreviated=False))
#pipe.to(device)
#print(torch.cuda.memory.memory_summary(device=device, abbreviated=False))

In [ ]:
BASE_SIZE = 512
INNER_SIZE = 256
OFFSET = (BASE_SIZE - INNER_SIZE) // 2 

try:
    input_image = Image.open("juta2.JPG").convert("RGB")
    width, height = input_image.size

    left = (width - INNER_SIZE) / 2
    top = (height - INNER_SIZE) / 2
    right = (width + INNER_SIZE) / 2
    bottom = (height + INNER_SIZE) / 2
    small_texture = input_image.crop((left, top, right, bottom))
    #small_texture = input_image.resize((256,256))

except:
    print("Immagine non trovata.")
    small_texture = Image.new("RGB", (INNER_SIZE, INNER_SIZE), "red")

small_texture.save("prova_1cropped_2.png")
display(small_texture)
#display(input_image)

In [ ]:
bg_hint = Image.new("RGB", (BASE_SIZE, BASE_SIZE), "white") 
canvas = bg_hint.copy()
canvas.paste(small_texture, (OFFSET, OFFSET))

mask = Image.new("L", (BASE_SIZE, BASE_SIZE), 255)
mask_center = Image.new("L", (INNER_SIZE, INNER_SIZE), 0)
mask.paste(mask_center, (OFFSET, OFFSET))
mask = mask.filter(ImageFilter.GaussianBlur(radius=3)) 

def create_guidance_image(source_img, target_size):
    tiled_bg = Image.new("RGB", (target_size, target_size))
    w, h = source_img.size
    tile_w, tile_h = 256, 256
    source_resized = source_img.resize((tile_w, tile_h))
    for i in range(0, target_size, tile_w):
        for j in range(0, target_size, tile_h):
            tiled_bg.paste(source_resized, (i, j))
    return tiled_bg


ip_adapter_image = create_guidance_image(small_texture, BASE_SIZE) 

ip_adapter_image.save("prova_2guidance_2.png")
display(ip_adapter_image)

In [ ]:
state["shift"] = (0, 0)
prompt = "Raw texture, regular pattern, flat colors, macro photography, close up, rough surface, hyperrealistic, top down view"
negative_prompt = "cartoon, drawing, painting, flat colors, vector, illustration, blurry, smooth, deformed, noisy"

seed = 1 
generator = torch.manual_seed(seed)

final_output = pipe(
    prompt=prompt,
    negative_prompt=negative_prompt,
    image=canvas, 
    mask_image=mask,
    generator=generator,
    ip_adapter_image=ip_adapter_image,
    num_inference_steps=50,
    guidance_scale=7.5,
    callback_on_step_end=callback, # chiama la callback alla fine di ogni step
    callback_on_step_end_tensor_inputs=["latents", "mask", "masked_image_latents"] # input callback
).images[0]
writer.close()

display(final_output)

final_output.save("result.png")

draw = ImageDraw.Draw(final_output) 
rect_coords = [OFFSET, OFFSET, OFFSET + INNER_SIZE, OFFSET + INNER_SIZE]
draw.rectangle(rect_coords, outline="green", width=4)

display(final_output)

final_output.save("result_cornice.png")
